<a href="https://colab.research.google.com/github/OkyereBiew/bayesian-credit-risk-simulator/blob/main/notebooks/05_model_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bayesian Credit Risk Simulator

## Model Evaluation & Performance Analysis

This notebook evaluates the Bayesian logistic regression model developed for probabilistic credit risk prediction.

The workflow includes:

- data preprocessing,
- Bayesian model reconstruction,
- posterior probability estimation,
- prediction generation,
- model evaluation metrics,
- and financial interpretation of classification performance.

In [ ]:
import pandas as pd
import numpy as np
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    roc_curve
)

In [ ]:
column_names = [
    "checking_account",
    "duration",
    "credit_history",
    "purpose",
    "credit_amount",
    "savings_account",
    "employment_duration",
    "installment_rate",
    "personal_status_sex",
    "other_debtors",
    "residence_duration",
    "property",
    "age",
    "other_installment_plans",
    "housing",
    "existing_credits",
    "job",
    "people_liable",
    "telephone",
    "foreign_worker",
    "target"
]

In [ ]:
df = pd.read_csv(
    "german.data",
    sep=" ",
    header=None,
    names=column_names
)

df.head()

TARGET VARIABLE TRANSFORMATION

In [ ]:
df["target"] = df["target"].map({
    1: 0,
    2: 1
})

df["target"].value_counts()

ENCODE CATEGORICAL VARIABLES

In [ ]:
categorical_columns = df.select_dtypes(
    include=["object"]
).columns

label_encoders = {}

for column in categorical_columns:

    encoder = LabelEncoder()

    df[column] = encoder.fit_transform(df[column])

    label_encoders[column] = encoder

SPLIT FEATURES & TARGET


In [ ]:
X = df.drop("target", axis=1)

y = df["target"]

SCALE FEATURES


In [ ]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

VERIFY DATA SHAPES

In [ ]:

print("Feature Matrix Shape:", X_scaled.shape)

print("Target Shape:", y.shape)

BUILD BAYESIAN LOGISTIC REGRESSION MODEL


In [ ]:
with pm.Model() as bayesian_logistic_model:

    # Priors for coefficients
    beta = pm.Normal(
        "beta",
        mu=0,
        sigma=1,
        shape=X_scaled.shape[1]
    )

    # Prior for intercept
    alpha = pm.Normal(
        "alpha",
        mu=0,
        sigma=1
    )

    # Linear predictor
    mu = alpha + pm.math.dot(X_scaled, beta)

    # Logistic transformation
    theta = pm.Deterministic(
        "theta",
        pm.math.sigmoid(mu)
    )

    # Likelihood
    likelihood = pm.Bernoulli(
        "likelihood",
        p=theta,
        observed=y
    )

    # MCMC sampling
    trace = pm.sample(
        draws=1000,
        tune=1000,
        target_accept=0.9,
        random_seed=42
    )

EXTRACT POSTERIOR DEFAULT PROBABILITIES


In [ ]:
posterior_probs = trace.posterior["theta"].mean(
    dim=["chain", "draw"]
).values

posterior_probs[:10]

CONVERT PROBABILITIES TO PREDICTIONS


In [ ]:
predictions = (
    posterior_probs >= 0.5
).astype(int)

predictions[:10]



COMPUTE MODEL ACCURACY

In [ ]:

accuracy = accuracy_score(
    y,
    predictions
)

print("Model Accuracy:", round(accuracy, 4))

## Accuracy Interpretation

Model accuracy measures the proportion of borrowers correctly classified by the Bayesian credit risk model.

Although accuracy is useful, financial risk systems also require careful evaluation of false positives and false negatives because different classification errors carry different financial consequences.

CONFUSION MATRIX


In [ ]:
cm = confusion_matrix(
    y,
    predictions
)

plt.figure(figsize=(6,5))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues"
)

plt.title("Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")

plt.show()

CLASSIFICATION REPORT


In [ ]:
print(
    classification_report(
        y,
        predictions
    )
)

ROC-AUC SCORE


In [ ]:
roc_auc = roc_auc_score(
    y,
    posterior_probs
)

print("ROC-AUC Score:", round(roc_auc, 4))

ROC CURVE


In [ ]:
fpr, tpr, thresholds = roc_curve(
    y,
    posterior_probs
)

plt.figure(figsize=(7,5))

plt.plot(fpr, tpr)

plt.plot(
    [0,1],
    [0,1],
    linestyle="--"
)

plt.title("ROC Curve")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")

plt.show()

POSTERIOR PROBABILITY DISTRIBUTION


In [ ]:
plt.figure(figsize=(8,5))

sns.histplot(
    posterior_probs,
    bins=30,
    kde=True
)

plt.title("Posterior Default Probability Distribution")
plt.xlabel("Default Probability")
plt.ylabel("Frequency")

plt.show()

CREDIT GROUP PROBABILITY DISTRIBUTION


In [ ]:
plt.figure(figsize=(8,5))

sns.histplot(
    posterior_probs[y == 0],
    label="Good Credit",
    kde=True
)

sns.histplot(
    posterior_probs[y == 1],
    label="Bad Credit",
    kde=True
)

plt.title("Posterior Default Probability by Credit Group")
plt.xlabel("Default Probability")
plt.ylabel("Frequency")

plt.legend()

plt.show()

In [ ]:
risk_results = pd.DataFrame({
    "Default_Probability": posterior_probs,
    "Actual_Target": y
})

def categorize_risk(probability):

    if probability < 0.30:
        return "Low Risk"

    elif probability < 0.60:
        return "Medium Risk"

    else:
        return "High Risk"

risk_results["Risk_Category"] = risk_results[
    "Default_Probability"
].apply(categorize_risk)

plt.figure(figsize=(8,5))

sns.boxplot(
    data=risk_results,
    x="Risk_Category",
    y="Default_Probability"
)

plt.title("Default Probability by Risk Category")
plt.xlabel("Risk Category")
plt.ylabel("Default Probability")

plt.show()

# Financial Interpretation

The Bayesian credit risk model demonstrates the ability to estimate borrower default probabilities while explicitly accounting for uncertainty in parameter estimation.

The confusion matrix highlights both successful classifications and prediction errors, which are critically important in lending analytics.

False negatives are especially costly in financial systems because incorrectly approving high-risk borrowers may result in loan defaults and financial losses.

The ROC-AUC score provides additional insight into the model’s ability to distinguish risky borrowers from safer applicants.

# Evaluation Summary

The Bayesian logistic regression model successfully generated probabilistic borrower risk predictions using posterior inference.

Evaluation metrics and visualizations demonstrate the model’s ability to distinguish between safer and riskier borrowers while incorporating uncertainty-aware estimation.

This workflow reflects real-world financial risk analytics practices involving probabilistic decision-making, model evaluation, and uncertainty quantification.